In [0]:
from pyspark.sql.functions import col, datediff, when

print("--- Criando Tabela Fato: Vendas & Logística ---")

# 1. Carregar as tabelas Silver
df_orders = spark.read.table("olist_portfolio.silver.orders")
df_items = spark.read.table("olist_portfolio.silver.order_items")
df_products = spark.read.table("olist_portfolio.silver.products")
df_sellers = spark.read.table("olist_portfolio.silver.sellers")

# 2. O Grande Join 
df_fact = (df_items.alias("i")
    .join(df_orders.alias("o"), "order_id")
    .join(df_products.alias("p"), "product_id")
    .join(df_sellers.alias("s"), "seller_id")
    .select(
        col("o.order_id"),
        col("o.customer_id"),
        col("i.product_id"),
        col("i.seller_id"),
        col("o.purchase_date"),
        col("o.order_status"),
        col("p.product_category_name"),
        col("s.seller_state").alias("seller_state"),
        col("i.price"),
        col("i.freight_value"),
        (col("i.price") + col("i.freight_value")).alias("total_value"),
        
      
        # Dias reais para entregar
        datediff(col("o.delivered_customer_date"), col("o.purchase_date")).alias("days_to_deliver"),
        
        # Flag de Atraso (Se entregou depois do estimado = 1, senão 0)
        when(col("o.delivered_customer_date") > col("o.estimated_delivery_date"), 1)
        .otherwise(0).alias("is_late_delivery")
    )
)

# 3. Salvar na Gold
df_fact.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("olist_portfolio.gold.fact_sales")
print("Sucesso: olist_portfolio.gold.fact_sales criada!")

In [0]:
from pyspark.sql.functions import max as spark_max, count, sum, lit, expr

print("--- Criando Dimensão: Clientes (Segmentação RFM) ---")

# Vamos usar a data máxima do dataset como "Hoje" para calcular a recência
# (Porque os dados são de 2018. Se usarmos current_date(), todo mundo vai parecer inativo há 5 anos)
max_date = spark.read.table("olist_portfolio.gold.fact_sales").select(spark_max("purchase_date")).collect()[0][0]

df_rfm = (spark.read.table("olist_portfolio.gold.fact_sales")
    .groupBy("customer_id")
    .agg(
        spark_max("purchase_date").alias("last_purchase"),
        count("order_id").alias("frequency"),
        sum("total_value").alias("monetary") 
    )
    .withColumn("recency_days", datediff(lit(max_date), col("last_purchase")))
    
    # Classificação Simples de Cliente (Regra de Negócio)
    .withColumn("customer_segment", 
        when((col("recency_days") < 30) & (col("monetary") > 500), "VIP")
        .when(col("recency_days") < 90, "Ativo")
        .when(col("recency_days") < 365, "Inativo Recente")
        .otherwise("Perdido")
    )
)

df_rfm.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("olist_portfolio.gold.dim_customer_rfm")
print("Sucesso: olist_portfolio.gold.dim_customer_rfm criada!")

In [0]:
print("--- Criando Tabela Analítica de Reviews ---")

df_reviews = spark.read.table("olist_portfolio.silver.reviews_nlp")
df_sales = spark.read.table("olist_portfolio.gold.fact_sales")

# Join para saber qual produto gerou o review ruim
df_gold_reviews = (df_reviews.alias("r")
    .join(df_sales.alias("s"), "order_id")
    .select(
        col("r.review_id"),
        col("s.order_id"),
        col("s.product_category_name"),
        col("r.sentiment_label"), 
        col("r.clean_text"),     
        col("s.purchase_date")
    )
)

df_gold_reviews.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("olist_portfolio.gold.reviews_analytics")
print("Sucesso: olist_portfolio.gold.reviews_analytics criada!")